## Data Leakage

Here’s a tiny, self-contained demo that shows data leakage inflating ROC–AUC by doing scaling + feature selection before cross-validation (leaky) vs doing them inside a Pipeline (clean).

In one run (with a fixed random seed), leakage inflated mean AUC by ~0.12.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

RANDOM_STATE = 7

# High-dimensional, low-signal setting where leakage effects are easy to see
X, y = make_classification(
    n_samples=300,
    n_features=500,
    n_informative=10,
    n_redundant=0,
    n_repeated=0,
    n_classes=2,
    class_sep=0.8,       # modest separability
    flip_y=0.10,         # label noise
    random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ---------------------------
# BAD (leaky) approach:
# Fit scaler + feature selection on the FULL dataset before CV
# ---------------------------
scaler = StandardScaler()
X_scaled_leaky = scaler.fit_transform(X)  # ❌ leakage: uses all data (incl. each fold's test part)

selector = SelectKBest(score_func=f_classif, k=25)
X_selected_leaky = selector.fit_transform(X_scaled_leaky, y)  # ❌ leakage: selector sees all labels

clf = LogisticRegression(max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE)
auc_leaky = cross_val_score(clf, X_selected_leaky, y, cv=cv, scoring="roc_auc")

# ---------------------------
# GOOD (non-leaky) approach:
# Everything inside Pipeline so each CV fold fits preprocessing only on its training split
# ---------------------------
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("select", SelectKBest(score_func=f_classif, k=25)),
    ("lr", LogisticRegression(max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE))
])
auc_clean = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")

print("5-fold CV ROC-AUC (LEAKY preprocessing done before CV):")
print("  fold AUCs:", np.round(auc_leaky, 3))
print("  mean ± std:", f"{auc_leaky.mean():.3f} ± {auc_leaky.std():.3f}")

print("\n5-fold CV ROC-AUC (CLEAN Pipeline, preprocessing inside CV):")
print("  fold AUCs:", np.round(auc_clean, 3))
print("  mean ± std:", f"{auc_clean.mean():.3f} ± {auc_clean.std():.3f}")

print("\nInflation (leaky mean - clean mean):", f"{(auc_leaky.mean() - auc_clean.mean()):.3f}")

5-fold CV ROC-AUC (LEAKY preprocessing done before CV):
  fold AUCs: [0.937 0.969 0.891 0.877 0.781]
  mean ± std: 0.891 ± 0.064

5-fold CV ROC-AUC (CLEAN Pipeline, preprocessing inside CV):
  fold AUCs: [0.726 0.851 0.794 0.828 0.663]
  mean ± std: 0.772 ± 0.069

Inflation (leaky mean - clean mean): 0.118


**Why the inflation happens here?**

SelectKBest(f_classif) uses labels to pick the “best” features. If you select features using the entire dataset (including the fold you later pretend is “test”), you’ve already “peeked” at the answers.